# Step 4 (v2) — Scaling Method Comparison for the GAN

**RESS 2025 — GAN-Conformal-RUL**

Trains the stage-conditioned WGAN-GP on **fold 1 only, 100 epochs**, under three
normalization schemes, and compares them on the quality gates. Whichever wins is used for
the full 5-fold run.

| Method | What it does |
|---|---|
| `log_clip` | log-transform amplitude/impulse/kurtosis, standardize, clip to ±5 |
| `robust` | median/IQR scaling on every feature, no log, no clip |
| `log_robust` | log-transform, then median/IQR scaling |

**Decision metrics (per method):**
1. Scaled data range — must fall within the generator's reachable band (~[-2, 5])
2. Wasserstein distance — should fall and flatten (diverging = the scaling defeats the GAN)
3. Real-vs-synthetic separability — closer to 50% is better
4. Diversity ratio — closer to 1 is better (low = mode collapse)

## 1. Setup

In [ ]:
import os, sys, shutil
os.chdir('/content')
REPO_PATH = '/content/RESS_2025_GAN_Conformal_RUL'
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone https://github.com/f-khadija-benzine/RESS_2025_GAN_Conformal_RUL.git {REPO_PATH}
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH); sys.path.insert(0, f'{REPO_PATH}/src')
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = f'{REPO_PATH}/figures'; os.makedirs(SAVE_DIR, exist_ok=True)
import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from data_loader import XJTUSYLoader
from health_indicator_v3 import HealthIndicatorPipeline
from windowing import prepare_all_folds, build_folds
from gan import StageGAN, GANConfig

TARGET = 2   # 0-indexed near-failure
METHODS = ['log_clip', 'robust', 'log_robust']

## 2. Rebuild Step 2 results (once, shared across methods)

In [ ]:
CANDIDATES = ['/content/drive/MyDrive/XJTU-SY',
              '/content/drive/MyDrive/XJTU-SY_Bearing_Datasets',
              '/content/drive/MyDrive/data/XJTU-SY']
DATA_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_ROOT, f'not found: {CANDIDATES}'
all_data = XJTUSYLoader(DATA_ROOT).load_all()
pipeline = HealthIndicatorPipeline(fpt_consecutive=5, fpt_min_relative_rise=0.20)
results = pipeline.process_all(all_data, verbose=False)
print('Step 2 done.')

## 3. Gate 1 — scaled range per method

The generator's `Linear` output naturally lands in roughly [-2, 5]. Any feature whose scaled
values fall well outside that band is unreachable, so the critic can separate real from fake
trivially and the GAN cannot converge. This gate is a fast pre-filter before any training.

In [ ]:
fold_data_by_method = {}
print(f"{'method':12s} {'overall range':22s} {'worst feature max':18s} {'unreachable (|x|>6)'}")
print('-'*75)
for m in METHODS:
    fd = prepare_all_folds(results, scaling_method=m, verbose=False)
    fold_data_by_method[m] = fd
    X = fd[0]['X_train']
    fmax = np.abs(X.reshape(-1, 20)).max(0)
    bad = np.where(fmax > 6)[0].tolist()
    print(f"{m:12s} [{X.min():7.2f}, {X.max():7.2f}]     {fmax.max():8.2f}          "
          f"{bad if bad else 'none'}")

## 4. Train a GAN under each method (fold 1, 100 epochs)

Same GAN, same fold, same everything except the input scaling. ~10 min each on a T4.

In [ ]:
EPOCHS = 100
gans_by_method = {}
for m in METHODS:
    print(f"\n{'='*60}\nSCALING = {m}\n{'='*60}")
    d = fold_data_by_method[m][0]   # fold 1
    cfg = GANConfig(epochs=EPOCHS, target_stage=3)
    gan = StageGAN(cfg)
    gan.fit(d['X_train'], d['y_stage_train'], verbose=True)
    gans_by_method[m] = gan

## 5. Gate 2 — Wasserstein convergence, side by side

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = {'log_clip': '#2166d4', 'robust': '#e34948', 'log_robust': '#2ca02c'}
for m in METHODS:
    ax.plot(gans_by_method[m].history['wasserstein'], label=m, color=colors[m], lw=1.4)
ax.axhline(0, color='gray', ls=':', lw=0.8)
ax.set_xlabel('epoch'); ax.set_ylabel('Wasserstein distance')
ax.set_title('Convergence by scaling method (lower & flatter = better)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_scaling_wasserstein_compare.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Gates 3 & 4 — separability and diversity, side by side

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from scipy.spatial.distance import pdist

print(f"{'method':12s} {'separability':14s} {'diversity ratio':16s} verdict")
print('-'*60)
summary = {}
for m in METHODS:
    gan = gans_by_method[m]
    d = fold_data_by_method[m][0]
    real = d['X_train'][d['y_stage_train'] == TARGET]
    n = min(len(real), 300)
    real_s = real[np.random.choice(len(real), n, replace=False)]
    syn = gan.sample(n, TARGET)

    # separability (~50% ideal)
    Xc = np.concatenate([real_s, syn]).reshape(2*n, -1)
    yc = np.r_[np.zeros(n), np.ones(n)]
    sep = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=0),
                          Xc, yc, cv=5).mean()
    # diversity (~1 ideal)
    div = pdist(syn.reshape(n, -1)).mean() / pdist(real_s.reshape(n, -1)).mean()

    score = abs(sep - 0.5) + abs(div - 1.0)   # lower is better
    summary[m] = {'sep': sep, 'div': div, 'score': score}
    verdict = 'good' if sep < 0.7 and div > 0.7 else 'poor'
    print(f"{m:12s} {sep:>6.1%}        {div:>6.2f}          {verdict}")

best = min(summary, key=lambda m: summary[m]['score'])
print(f"\nBest scaling method: {best}")
print('(lowest combined distance from ideal separability=50% and diversity=1.0)')

## 7. Decision

Read the three gates together:
- **Range** (Gate 1): a method whose features fall outside ~[-2, 5] is disqualified regardless of the rest.
- **Wasserstein** (Gate 2): must fall and flatten, not diverge.
- **Separability + diversity** (Gates 3–4): among methods that pass 1–2, pick the best.

The winning `scaling_method` is then passed to `prepare_all_folds(...)` in the full Step 4
run (`04_train_gan.ipynb`) and recorded in Section 4.3 as the chosen normalization, with the
comparison reported as justification.